# 04. Single-Mode Squeezed Vacuum & Quadrature Noise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_1_states_of_light/04_squeezed_states_and_quadrature_noise.ipynb)

* **Module:** 1 — Single-Mode Quantum States of Light
* **Difficulty:** Intermediate / Advanced
* **Estimated Time:** 50 minutes
* **Prerequisites:** Notebook 1.1 (Fock States & Ladder Operators), Notebook 1.2 (Coherent States & Phase Space).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Formulate the Squeeze Operator** $\hat{S}(\xi) = \exp\left[\frac{1}{2}(\xi^*\hat{a}^2 - \xi\hat{a}^{\dagger 2})\right]$ and generate Squeezed Vacuum States $|\xi\rangle = \hat{S}(\xi)|0\rangle$ in QuTiP.
2. **Apply the Bogoliubov transformation** to ladder operators: $\hat{S}^\dagger\hat{a}\hat{S} = \hat{a}\cosh r - \hat{a}^\dagger e^{i\theta}\sinh r$.
3. **Demonstrate sub-shot-noise quantum noise reduction**: prove that one quadrature variance drops below the vacuum limit ($\langle (\Delta X_1)^2 \rangle = \frac{1}{4}e^{-2r} < 0.25$) while preserving minimum uncertainty $\Delta X_1 \Delta X_2 = 1/4$.
4. **Quantify squeezing in decibels (dB)**: calculate noise suppression relative to the vacuum shot-noise level ($\text{dB} \approx 8.686\,r$), connecting to gravitational wave detection in LIGO.
5. **Analyze the Even-Fock photon statistics**: prove that squeezed vacuum is composed strictly of photon pairs ($P(2m+1) \equiv 0$), exhibiting an oscillatory photon number distribution.
6. **Map phase space noise ellipses**: visualize the rotation and elongation of the noise ellipse as a function of squeeze factor $r$ and angle $\theta$.


---
## 0. Setup & Dependencies

We will use **QuTiP (Quantum Toolbox in Python)** alongside NumPy and Matplotlib.


In [ ]:
# Install dependencies in Google Colab (uncomment if running in Colab)
# !pip install -q qutip numpy scipy matplotlib ipywidgets

import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import math

# QuTiP import
try:
    import qutip as qt
    from qutip import basis, destroy, create, num, squeeze, qeye
    has_qutip = True
    print(f"QuTiP version {qt.__version__} initialized.")
except ImportError:
    has_qutip = False
    print("QuTiP not installed locally. Using internal NumPy fallback.")

# Configure clean, publication-ready plotting aesthetics
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})


---
## 1. Physical Motivation: Beyond the Vacuum Shot-Noise Limit

In coherent laser states and the vacuum, quantum noise is isotropic across all phases:

$$\Delta X_1 = \Delta X_2 = \frac{1}{2} \implies \langle (\Delta X_1)^2 \rangle = \langle (\Delta X_2)^2 \rangle = \frac{1}{4}$$

This irreducible floor of $1/4$ is known as the **Standard Quantum Limit (SQL)** or **vacuum shot noise**.

However, the Heisenberg uncertainty relation only restricts the **product** of quadrature uncertainties:

$$\Delta X_1 \, \Delta X_2 \ge \frac{1}{4}$$

It does **not** forbid $\Delta X_1 < 1/2$, provided that the conjugate uncertainty $\Delta X_2$ expands proportionately! 
A state in which one quadrature variance is reduced below the vacuum level ($< 1/4$) is called a **Quadrature Squeezed State**.

### Real-World Impact: Gravitational Wave Detection (LIGO)
In laser interferometer gravitational wave detectors like LIGO and Virgo, kilometer-scale Michelson interferometers measure tiny mirror displacements ($\sim 10^{-19}\text{ m}$) caused by passing gravitational waves. 

At high optical frequencies ($> 100\text{ Hz}$), the detector sensitivity is limited by photon shot noise entering through the unused (dark) output port. By injecting **squeezed vacuum light** whose quiet quadrature aligns with the detector readout, LIGO achieves up to $3\text{ dB}$ to $10\text{ dB}$ of quantum noise reduction, vastly increasing the observable volume of the universe!


---
## 2. The Squeeze Operator $\hat{S}(\xi)$ & Bogoliubov Transformations

The unitary single-mode **squeeze operator** is defined as:

$$\hat{S}(\xi) = \exp\left[ \frac{1}{2}\left( \xi^* \hat{a}^2 - \xi \hat{a}^{\dagger 2} \right) \right]$$

where $\xi = r e^{i\theta}$ is the complex squeezing parameter:
* $r \ge 0$ is the **squeezing parameter** (amplitude of noise reduction).
* $\theta \in [0, 2\pi]$ is the **squeezing angle** (determines which quadrature is quieted).

Notice that the exponent involves quadratic terms ($\hat{a}^2$ and $\hat{a}^{\dagger 2}$). In nonlinear optics, this Hamiltonian arises from a second-order nonlinear crystal $\chi^{(2)}$ driven by a strong pump laser at frequency $2\omega$, converting pump photons into correlated **photon pairs** at frequency $\omega$ (Optical Parametric Amplification).

### The Bogoliubov Transformation
Applying unitary similarity transformations to the annihilation and creation operators:

$$\hat{S}^\dagger(\xi)\,\hat{a}\,\hat{S}(\xi) = \hat{a}\cosh(r) - \hat{a}^\dagger e^{i\theta}\sinh(r)$$

$$\hat{S}^\dagger(\xi)\,\hat{a}^\dagger\,\hat{S}(\xi) = \hat{a}^\dagger\cosh(r) - \hat{a} e^{-i\theta}\sinh(r)$$

Because $\hat{a}$ mixes linearly with $\hat{a}^\dagger$, squeezing fundamentally redefines the mode's excitation structure.


In [ ]:
def construct_squeezed_vacuum(N: int, r: float, theta: float = 0.0):
    """
    Constructs the squeezed vacuum state |xi> = S(xi)|0> in dimension N.
    """
    xi = r * np.exp(1j * theta)
    if has_qutip:
        return squeeze(N, xi) * basis(N, 0)
    else:
        # NumPy fallback via Bogoliubov transformation and Fock expansion
        # |xi> = (1/sqrt(cosh r)) * sum_m (sqrt((2m)!)/(2^m * m!)) * (-exp(i*theta)*tanh r)^m |2m>
        psi = np.zeros(N, dtype=complex)
        prefactor = 1.0 / np.sqrt(np.cosh(r))
        tanh_term = -np.exp(1j * theta) * np.tanh(r)
        
        for m in range(N // 2):
            idx = 2 * m
            if idx < N:
                coeff = math.sqrt(float(math.factorial(idx))) / ((2**m) * float(math.factorial(m)))
                psi[idx] = prefactor * coeff * (tanh_term**m)
                
        # Normalize to handle truncation
        norm = np.linalg.norm(psi)
        return psi / norm

# Test state generation for r = 0.8, theta = 0
N_test = 40
r_val = 0.8
psi_sq = construct_squeezed_vacuum(N_test, r=r_val, theta=0.0)

print(f"Generated Squeezed Vacuum state for r = {r_val}")
if has_qutip:
    print(f"Norm of state: {(psi_sq.dag() * psi_sq).tr().real:.8f}")
else:
    print(f"Norm of state: {np.sum(np.abs(psi_sq)**2):.8f}")


---
## 3. Quadrature Variances: Sub-Shot-Noise Reduction

Let us compute the variances of the field quadratures $\hat{X}_1 = \frac{1}{2}(\hat{a} + \hat{a}^\dagger)$ and $\hat{X}_2 = \frac{1}{2i}(\hat{a} - \hat{a}^\dagger)$ for the squeezed vacuum $|\xi\rangle = \hat{S}(r)|0\rangle$ with squeezing angle $\theta = 0$:

1. **Mean values:**
   $$\langle \xi | \hat{X}_1 | \xi \rangle = 0, \quad \langle \xi | \hat{X}_2 | \xi \rangle = 0$$
   The squeezed vacuum has zero mean electric and magnetic fields.

2. **Quadrature Variances:**
   Using the Bogoliubov transformation:
   $$\hat{S}^\dagger(r)\hat{X}_1\hat{S}(r) = \frac{1}{2}\left( \hat{a}(\cosh r - \sinh r) + \hat{a}^\dagger(\cosh r - \sinh r) \right) = e^{-r}\hat{X}_1$$
   $$\hat{S}^\dagger(r)\hat{X}_2\hat{S}(r) = \frac{1}{2i}\left( \hat{a}(\cosh r + \sinh r) - \hat{a}^\dagger(\cosh r + \sinh r) \right) = e^{+r}\hat{X}_2$$

Taking the expectation value with respect to the vacuum $|0\rangle$:

$$\langle (\Delta X_1)^2 \rangle = \langle 0 | \left(e^{-r}\hat{X}_1\right)^2 | 0 \rangle = \frac{1}{4} e^{-2r}$$

$$\langle (\Delta X_2)^2 \rangle = \langle 0 | \left(e^{+r}\hat{X}_2\right)^2 | 0 \rangle = \frac{1}{4} e^{+2r}$$

### The Minimum Uncertainty Squeezed State
Multiplying the two uncertainties:

$$\Delta X_1 \Delta X_2 = \left(\frac{1}{2}e^{-r}\right) \left(\frac{1}{2}e^{+r}\right) = \frac{1}{4}$$

* **Noise along $X_1$ is exponentially suppressed** by factor $e^{-2r}$.
* **Noise along $X_2$ is exponentially amplified** by factor $e^{+2r}$.
* The uncertainty product is **strictly preserved at the minimum quantum bound ($1/4$)**!

### Measuring Squeezing in Decibels (dB)
Experimentalists quantify noise reduction in logarithmic units (decibels) relative to the vacuum shot noise:

$$\text{Squeezing (dB)} = -10 \log_{10}\left( \frac{\langle (\Delta X_1)^2 \rangle}{\langle (\Delta X_1)^2 \rangle_{\text{vac}}} \right) = -10 \log_{10}\left( e^{-2r} \right) = \frac{20}{\ln 10} r \approx 8.686 \, r \text{ dB}$$

* $r = 0.35 \implies \approx 3\text{ dB}$ (noise reduced by a factor of $2$).
* $r = 0.69 \implies \approx 6\text{ dB}$ (noise reduced by a factor of $4$).
* $r = 1.15 \implies \approx 10\text{ dB}$ (noise reduced by a factor of $10$).


In [ ]:
def calculate_squeezing_vs_r():
    r_range = np.linspace(0, 1.4, 60)
    N_dim = 60
    
    var_x1_num = []
    var_x2_num = []
    
    for r in r_range:
        psi = construct_squeezed_vacuum(N_dim, r=r, theta=0.0)
        
        if has_qutip:
            a_op = destroy(N_dim)
            adag_op = create(N_dim)
            X1 = 0.5 * (a_op + adag_op)
            X2 = -0.5j * (a_op - adag_op)
            
            v1 = (qt.expect(X1*X1, psi) - qt.expect(X1, psi)**2).real
            v2 = (qt.expect(X2*X2, psi) - qt.expect(X2, psi)**2).real
        else:
            a_op = np.zeros((N_dim, N_dim), dtype=complex)
            for k in range(1, N_dim):
                a_op[k-1, k] = np.sqrt(k)
            adag_op = a_op.conj().T
            X1 = 0.5 * (a_op + adag_op)
            X2 = -0.5j * (a_op - adag_op)
            
            v1 = (psi.conj() @ (X1 @ (X1 @ psi))).real
            v2 = (psi.conj() @ (X2 @ (X2 @ psi))).real
            
        var_x1_num.append(v1)
        var_x2_num.append(v2)
        
    var_x1_exact = 0.25 * np.exp(-2.0 * r_range)
    var_x2_exact = 0.25 * np.exp(+2.0 * r_range)
    sq_db = -10.0 * np.log10(np.array(var_x1_num) / 0.25)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    
    # 1. Variance vs r
    ax1.plot(r_range, var_x1_num, 'b-', lw=2.2, label=r"Squeezed Quadrature $\langle (\Delta X_1)^2 \rangle$")
    ax1.plot(r_range, var_x2_num, 'r-', lw=2.2, label=r"Anti-Squeezed Quadrature $\langle (\Delta X_2)^2 \rangle$")
    ax1.axhline(0.25, color="gray", ls="--", lw=1.5, label="Vacuum Shot Noise (0.25)")
    ax1.set_title("Quadrature Variances vs. Squeeze Parameter $r$")
    ax1.set_xlabel("Squeeze Parameter $r$")
    ax1.set_ylabel("Variance")
    ax1.set_yscale("log")
    ax1.legend(loc="center left")
    
    # 2. Squeezing in dB
    ax2.plot(r_range, sq_db, 'g-', lw=2.2, label="Quantum Noise Reduction (dB)")
    ax2.plot(r_range, 8.686 * r_range, 'k--', lw=1.5, label=r"Analytical $8.686\,r\mathrm{\ dB}$")
    ax2.axhline(3.0, color="orange", ls=":", label="3 dB (50% noise reduction)")
    ax2.axhline(10.0, color="red", ls=":", label="10 dB (90% noise reduction)")
    ax2.set_title("Noise Reduction in Decibels (dB)")
    ax2.set_xlabel("Squeeze Parameter $r$")
    ax2.set_ylabel("Squeezing (dB)")
    ax2.legend(loc="upper left")
    
    plt.tight_layout()
    plt.show()

calculate_squeezing_vs_r()


---
## 4. Phase Space Geometry: Squeezed Noise Ellipses

In the complex amplitude plane $(X_1, X_2)$, a coherent state is represented by an isotropic circle of radius $\Delta X_1 = \Delta X_2 = 1/2$.

For a squeezed state, the noise distribution deforms into an **ellipse**:
* Semi-minor axis: $\Delta X_{\text{min}} = \frac{1}{2}e^{-r}$ (compressed noise).
* Semi-major axis: $\Delta X_{\text{max}} = \frac{1}{2}e^{+r}$ (dilated noise).
* Orientation angle: The axis of squeezing is rotated by $\theta/2$ relative to the $X_1$ axis.

Because the area of the ellipse is $\pi (\Delta X_1)(\Delta X_2) = \frac{\pi}{4} = \text{const}$, squeezing **preserves the phase space area** (symplectic invariance / Liouville's theorem in phase space)!


In [ ]:
def plot_squeezed_noise_ellipses():
    r_values = [0.0, 0.4, 0.8, 1.2]
    theta_rot = np.pi / 4.0  # 45 degree squeezing angle -> ellipse rotated by 22.5 deg
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
    t = np.linspace(0, 2*np.pi, 300)
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    
    # 1. Amplitude-squeezed states (theta = 0, aligned with X1)
    for i, r in enumerate(r_values):
        rx = 0.5 * np.exp(-r)
        ry = 0.5 * np.exp(+r)
        ax1.plot(rx * np.cos(t), ry * np.sin(t), color=colors[i], lw=2.0,
                 label=rf"$r={r:.1f}$ ($\Delta X_1={rx:.2f}, \Delta X_2={ry:.2f}$)")
        ax1.fill(rx * np.cos(t), ry * np.sin(t), color=colors[i], alpha=0.1)
        
    ax1.axhline(0, color="gray", lw=1)
    ax1.axvline(0, color="gray", lw=1)
    ax1.set_aspect("equal")
    ax1.set_xlim(-2.0, 2.0)
    ax1.set_ylim(-2.0, 2.0)
    ax1.set_title(r"Squeezed Vacuum Ellipses ($\theta = 0$, Aligned with $X_1$)")
    ax1.set_xlabel(r"Quadrature $X_1$")
    ax1.set_ylabel(r"Quadrature $X_2$")
    ax1.legend(loc="upper left")
    
    # 2. Rotated squeezed states (varying theta for fixed r = 0.8)
    r_fixed = 0.8
    thetas = [0.0, np.pi/2, np.pi, 3*np.pi/2]
    theta_labels = [r"$\theta = 0$", r"$\theta = \pi/2$", r"$\theta = \pi$", r"$\theta = 3\pi/2$"]
    
    for j, th in enumerate(thetas):
        rx = 0.5 * np.exp(-r_fixed)
        ry = 0.5 * np.exp(+r_fixed)
        # Unrotated coordinates
        x_raw = rx * np.cos(t)
        y_raw = ry * np.sin(t)
        # Rotation by angle theta/2
        phi_rot = th / 2.0
        x_rot = x_raw * np.cos(phi_rot) - y_raw * np.sin(phi_rot)
        y_rot = x_raw * np.sin(phi_rot) + y_raw * np.cos(phi_rot)
        
        ax2.plot(x_rot, y_rot, color=colors[j], lw=2.0, label=theta_labels[j])
        ax2.fill(x_rot, y_rot, color=colors[j], alpha=0.1)
        
    ax2.axhline(0, color="gray", lw=1)
    ax2.axvline(0, color="gray", lw=1)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.8, 1.8)
    ax2.set_ylim(-1.8, 1.8)
    ax2.set_title(rf"Rotated Noise Ellipses ($r = {r_fixed}$, Rotation by $\theta/2$)")
    ax2.set_xlabel(r"Quadrature $X_1$")
    ax2.set_ylabel(r"Quadrature $X_2$")
    ax2.legend(loc="upper left")
    
    plt.tight_layout()
    plt.show()

plot_squeezed_noise_ellipses()


---
## 5. Photon Number Statistics: Exclusive Even-Fock Superpositions

What does a squeezed vacuum state look like in terms of photon numbers?

Using the operator identity $\hat{S}(\xi) = \exp[\frac{1}{2}(\xi^*\hat{a}^2 - \xi\hat{a}^{\dagger 2})]$, the creation operator occurs **strictly in pairs** $\hat{a}^{\dagger 2}$. Applying this to the vacuum state $|0\rangle$:

$$|\xi\rangle = \frac{1}{\sqrt{\cosh r}} \sum_{m=0}^{\infty} \frac{\sqrt{(2m)!}}{2^m m!} \left( -e^{i\theta}\tanh r \right)^m |2m\rangle$$

### Distinctive Features of Squeezed Vacuum Photons
1. **Zero Odd-Photon Populations:**
   $$P(2m+1) \equiv 0 \quad \text{for all } m \ge 0$$
   Photons are generated strictly in pairs! You will never detect an odd number of photons ($1, 3, 5, \dots$) in a pure squeezed vacuum state.
2. **Mean Photon Number:**
   $$\bar{n} = \langle \xi | \hat{n} | \xi \rangle = \sinh^2(r)$$
   Even though the state has zero average electric field ($\langle \hat{X}_1 \rangle = 0$), it carries non-zero energy $\hbar\omega \sinh^2(r)$!
3. **Photon Number Variance:**
   $$\langle (\Delta n)^2 \rangle = 2\sinh^2(r)\cosh^2(r) = 2\bar{n}(\bar{n} + 1)$$
   The number fluctuations are **super-Poissonian** ($\langle (\Delta n)^2 \rangle > \bar{n}$). Squeezing light reduces quadrature noise below the shot-noise limit at the cost of wildly fluctuating photon numbers!


In [ ]:
def plot_even_fock_statistics():
    r_vals = [0.4, 0.8, 1.2]
    N_fock = 24
    fock_axis = np.arange(N_fock)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    
    for idx, r in enumerate(r_vals):
        ax = axes[idx]
        psi_sq = construct_squeezed_vacuum(N=N_fock + 15, r=r)
        
        if has_qutip:
            probs = [qt.expect(basis(N_fock + 15, n).proj(), psi_sq) for n in fock_axis]
        else:
            probs = np.abs(psi_sq[:N_fock])**2
            
        n_bar = np.sinh(r)**2
        var_n = 2.0 * n_bar * (n_bar + 1.0)
        
        colors = ["#1f77b4" if n % 2 == 0 else "#d62728" for n in fock_axis]
        ax.bar(fock_axis, probs, color=colors, width=0.65, edgecolor="black", lw=0.6)
        ax.axvline(n_bar, color="green", ls="--", lw=2, label=rf"$\bar{{n}} = \sinh^2(r) = {n_bar:.2f}$")
        
        ax.set_title(rf"Squeezed Vacuum ($r = {r}$, $\Delta n = {np.sqrt(var_n):.2f}$)")
        ax.set_xlabel("Photon Number $n$")
        ax.set_ylabel("Probability $P(n)$")
        ax.set_xticks(np.arange(0, N_fock, 2))
        ax.set_ylim(0, max(0.4, max(probs)*1.15))
        ax.legend()
        
    plt.tight_layout()
    plt.show()

plot_even_fock_statistics()


> **Physical Insight:** Notice the dramatic alternating pattern: **even Fock numbers ($0, 2, 4, \dots$) have high probability, while odd Fock numbers ($1, 3, 5, \dots$) are strictly zero!** This pairwise photon generation is the direct microscopic signature of parametric down-conversion ($2\omega \to \omega + \omega$).


---
## 6. Interactive Playground: Squeezing & Phase Space Ellipse Explorer

Use the interactive sliders below to explore how the **squeeze factor $r$** and **angle $\theta$** reshape the noise ellipse in phase space and control the photon pair distribution.


In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    has_widgets = True
except ImportError:
    has_widgets = False

def interactive_squeeze_explorer(r=0.8, theta_factor=0.0):
    theta = theta_factor * np.pi
    n_bar = np.sinh(r)**2
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 1. Phase Space Ellipse
    t = np.linspace(0, 2*np.pi, 300)
    rx = 0.5 * np.exp(-r)
    ry = 0.5 * np.exp(+r)
    
    # Rotate by theta/2
    phi_rot = theta / 2.0
    x_rot = (rx * np.cos(t)) * np.cos(phi_rot) - (ry * np.sin(t)) * np.sin(phi_rot)
    y_rot = (rx * np.cos(t)) * np.sin(phi_rot) + (ry * np.sin(t)) * np.cos(phi_rot)
    
    # Plot vacuum shot-noise circle for reference
    ax1.plot(0.5 * np.cos(t), 0.5 * np.sin(t), 'k--', lw=1.2, alpha=0.6, label="Vacuum Circle (0.5)")
    ax1.plot(x_rot, y_rot, color="#d62728", lw=2.2, label=rf"Squeezed Ellipse ($r={r:.2f}$)")
    ax1.fill(x_rot, y_rot, color="#d62728", alpha=0.15)
    
    sq_db = 8.686 * r
    ax1.axhline(0, color="gray", lw=1)
    ax1.axvline(0, color="gray", lw=1)
    ax1.set_aspect("equal")
    ax1.set_xlim(-2.2, 2.2)
    ax1.set_ylim(-2.2, 2.2)
    ax1.set_title(rf"Phase Space Ellipse ({sq_db:.1f} dB Squeezing)")
    ax1.set_xlabel(r"Quadrature $X_1$")
    ax1.set_ylabel(r"Quadrature $X_2$")
    ax1.legend(loc="upper left")
    
    # 2. Photon Statistics
    N_pts = 16
    fock_k = np.arange(N_pts)
    probs = np.zeros(N_pts)
    for m in range(N_pts // 2):
        idx = 2 * m
        coeff = math.sqrt(float(math.factorial(idx))) / ((2**m) * float(math.factorial(m)))
        probs[idx] = (1.0 / np.cosh(r)) * (coeff**2) * (np.tanh(r)**(2*m))
        
    ax2.bar(fock_k, probs, color="#1f77b4", width=0.65, edgecolor="black")
    ax2.axvline(n_bar, color="#d62728", ls="--", lw=2, label=rf"Mean $\bar{{n}} = {n_bar:.2f}$")
    ax2.set_title("Pairwise Photon Statistics $P(n)$")
    ax2.set_xlabel("Photon Number $n$")
    ax2.set_ylabel("Probability")
    ax2.set_xticks(np.arange(0, N_pts, 2))
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

if has_widgets:
    interact(
        interactive_squeeze_explorer,
        r=FloatSlider(min=0.0, max=1.5, step=0.05, value=0.75, description="Squeeze r:"),
        theta_factor=FloatSlider(min=0.0, max=2.0, step=0.05, value=0.0, description="Angle \u03b8/\u03c0:")
    )
else:
    print("Interactive sliders active in Jupyter/Colab. Displaying baseline r = 0.75:")
    interactive_squeeze_explorer(r=0.75, theta_factor=0.0)


---
## 7. Hands-On Exercises & Guided Challenges

### Exercise 1: Squeezing Parameters for Advanced LIGO
Advanced LIGO upgrades target approximately $10\text{ dB}$ of quantum noise reduction at high frequencies.
1. What value of the squeezing parameter $r$ is required to achieve exactly $10\text{ dB}$ of squeezing?
2. What is the mean photon number $\bar{n}$ carried by this squeezed vacuum beam?
3. By what factor is the anti-squeezed quadrature variance amplified?

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Solution:**
1. Using $\text{dB} = \frac{20}{\ln 10} r \approx 8.686\, r$:
   $$r = \frac{10}{8.686} = \frac{\ln(10)}{2} \approx 1.1513$$
2. Mean photon number:
   $$\bar{n} = \sinh^2(r) = \sinh^2(1.1513) \approx (1.424)^2 \approx 2.03 \text{ photons}$$
   Notice that a modest average count of $\sim 2$ photons provides a $10$-fold noise suppression!
3. The anti-squeezed quadrature variance is amplified by factor $e^{+2r} = e^{2.3026} = 10.0$ (a factor of $10$).
</details>

---

### Exercise 2: Analytical Derivation of Photon Pair Probabilities
Using the squeezed state formula:
$$P(2m) = \frac{1}{\cosh r} \frac{(2m)!}{(2^m m!)^2} (\tanh r)^{2m}$$
Verify that for $m=0$ (the probability of detecting zero photons), $P(0) = \frac{1}{\cosh r}$.
Show that for $r = 1.0$, $P(0) \approx 0.648$, meaning the state has a $64.8\%$ chance of being completely empty vacuum despite having mean photon number $\bar{n} = \sinh^2(1.0) \approx 1.381$!


---
## 8. Summary & What's Next

### Key Takeaways
1. **Squeezed Light Beats Shot Noise:** The squeeze operator $\hat{S}(\xi)$ reduces quantum fluctuations in one quadrature below the standard quantum limit ($\langle (\Delta X_1)^2 \rangle = \frac{1}{4}e^{-2r} < 1/4$), while anti-squeezing the conjugate quadrature to preserve $\Delta X_1 \Delta X_2 = 1/4$.
2. **Bogoliubov Transformation:** Squeezing mixes ladder operators: $\hat{S}^\dagger\hat{a}\hat{S} = \hat{a}\cosh r - \hat{a}^\dagger e^{i\theta}\sinh r$.
3. **Decibel Scaling:** Noise reduction in dB scales as $\text{dB} \approx 8.686\,r$. A squeeze parameter of $r \approx 1.15$ yields $10\text{ dB}$ (90% noise reduction), crucial for gravitational wave interferometry.
4. **Strict Photon Pair Statistics:** Squeezed vacuum contains exclusively even photon numbers ($P(2m+1) \equiv 0$), with super-Poissonian variance $\langle (\Delta n)^2 \rangle = 2\bar{n}(\bar{n}+1)$.
5. **Phase Space Ellipses:** In $(X_1, X_2)$, squeezed states form elliptical noise distributions rotated by $\theta/2$, with preserved phase-space area $\pi/4$.

---

### What's Next?
In **Notebook 1.4 (`04_displaced_squeezed_states.ipynb`)**, we will:
* Combine displacement and squeezing: analyze **Displaced Squeezed States** $\hat{D}(\alpha)\hat{S}(\xi)|0\rangle$.
* Study operator ordering differences: $\hat{D}(\alpha)\hat{S}(\xi)$ vs. $\hat{S}(\xi)\hat{D}(\alpha)$.
* Discover **Quantum Interference in Phase Space**: how displacing a squeezed state generates wild quantum oscillations in the photon number distribution!

---
### References & Recommended Reading
* **Carlton M. Caves**, *Quantum-mechanical noise in an interferometer*, Phys. Rev. D 23, 1693 (1981).
* **R. E. Slusher et al.**, *Observation of Squeezed States Generated by Four-Wave Mixing in an Optical Cavity*, Phys. Rev. Lett. 55, 2409 (1985) *(First experimental observation)*.
* **LIGO Scientific Collaboration**, *A gravitational wave observatory operating beyond the quantum shot-noise limit*, Nature Physics 7, 962 (2011).
